# RAG Benchmark — Generation Stage on Colab

Runs the optional generation and answer-scoring stages of the COMP702 RAG
benchmark on a Colab GPU. Retrieval is already complete and is **not** repeated
here: every model answers the byte-identical context that the retrieval
benchmark froze, so differences between models are attributable to the model.

**Repository:** https://github.com/mahesh062003/ai-benchmark

## Before you start

Your Google Drive folder `AI Benchmark` must contain:

| Path | What | Size |
|---|---|---|
| `AI Benchmark/datasets/` | the six raw datasets | ~1 GB |
| `AI Benchmark/artifacts/corpora/` | chunk texts used to build prompts | ~352 MB |
| `AI Benchmark/artifacts/benchmark.sqlite` | the completed retrieval run | ~291 MB |

`artifacts/indexes/` is **not** needed — retrieval has already run.

Upload `artifacts/corpora/` and `artifacts/benchmark.sqlite` from
`C:\AI BENCHMARK\artifacts\` if they are not in Drive yet.

## Why this notebook does not write SQLite directly to Drive

Google Drive is mounted through FUSE, which does not implement the file locking
SQLite relies on. Writing a database there during a long run risks
`database is locked` errors and, in the worst case, a corrupted file holding
every result you have.

So the notebook copies the database to Colab's local disk, runs against it
there, and **copies it back to Drive every 10 minutes** using SQLite's online
backup API, which is safe against a database being written to. Nothing is lost
if the session dies: the Drive copy is at most 10 minutes behind, and the run
resumes from whatever it contains.

## Runtime

Use an **L4** if you have one. Roughly 15 hours for 7,200 generations plus
scoring, which exceeds Colab's 12-hour cap, so expect two sessions. Re-running
this notebook resumes rather than restarting.

## 1 · Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

# 7-9B models at fp16 need roughly 18 GB. On a 16 GB card Ollama will fall back
# to partial CPU offload, which still works but is several times slower.

## 2 · Mount Drive and verify the layout

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

DRIVE = Path('/content/drive/MyDrive/AI Benchmark')
DRIVE_DATASETS = DRIVE / 'datasets'
DRIVE_ARTIFACTS = DRIVE / 'artifacts'

print(f'Drive folder: {DRIVE}')
if not DRIVE.exists():
    raise SystemExit(
        f'{DRIVE} not found. Check the folder name is exactly "AI Benchmark" '
        'and that it sits at the top level of My Drive.'
    )

required = {
    'datasets/':                DRIVE_DATASETS,
    'artifacts/corpora/':       DRIVE_ARTIFACTS / 'corpora',
    'artifacts/benchmark.sqlite': DRIVE_ARTIFACTS / 'benchmark.sqlite',
}
missing = []
for label, path in required.items():
    ok = path.exists()
    print(f'  {"OK  " if ok else "MISS"}  {label}')
    if not ok:
        missing.append(label)
if missing:
    raise SystemExit(
        'Missing from Drive: ' + ', '.join(missing) +
        '\nUpload them from C:\\AI BENCHMARK\\ before continuing.'
    )

## 3 · Clone the repository

In [ ]:
import subprocess
from pathlib import Path

REPO = 'https://github.com/mahesh062003/ai-benchmark.git'
PROJECT = Path('/content/ai-benchmark')

if PROJECT.exists():
    print('updating existing clone')
    subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, str(PROJECT)], check=True)

%cd /content/ai-benchmark
!git log --oneline -1

## 4 · Install Python dependencies

In [ ]:
# Colab already ships torch, numpy and pandas; pip will leave those alone.
!pip install -q -r /content/ai-benchmark/requirements.txt 2>&1 | tail -5
print('dependencies ready')

## 5 · Install Ollama and pull the four models

About **19 GB** of model weights. They are stored on Colab's local disk, not
Drive, because loading a model over FUSE is far slower than re-downloading it.
Expect 10–20 minutes on the first run of each session.

In [ ]:
import os
import subprocess
import time

import requests

os.environ['OLLAMA_MODELS'] = '/content/ollama_models'
os.makedirs('/content/ollama_models', exist_ok=True)

if subprocess.run(['which', 'ollama'], capture_output=True).returncode != 0:
    print('installing ollama...')
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
else:
    print('ollama already installed')

# Start the server detached; it must outlive this cell.
subprocess.Popen(
    ['ollama', 'serve'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
    env={**os.environ},
)

for attempt in range(60):
    try:
        if requests.get('http://localhost:11434/api/tags', timeout=2).ok:
            print('ollama server is up')
            break
    except Exception:
        pass
    time.sleep(2)
else:
    raise SystemExit('ollama did not start; re-run this cell')

MODELS = ['llama3.1', 'gemma2', 'mistral', 'qwen2.5']
for model in MODELS:
    print(f'--- pulling {model} ---')
    subprocess.run(['ollama', 'pull', model], check=True)

!ollama list

## 6 · Stage artifacts onto local disk

Copied from Drive so the database is written on a real filesystem. Only the
corpora and the database are needed; the FAISS indexes are not.

In [ ]:
import shutil
from pathlib import Path

LOCAL_ARTIFACTS = Path('/content/artifacts')
LOCAL_ARTIFACTS.mkdir(parents=True, exist_ok=True)
(LOCAL_ARTIFACTS / 'results').mkdir(exist_ok=True)

local_db = LOCAL_ARTIFACTS / 'benchmark.sqlite'
local_corpora = LOCAL_ARTIFACTS / 'corpora'

if not local_corpora.exists():
    print('copying corpora from Drive (a few minutes)...')
    shutil.copytree(DRIVE_ARTIFACTS / 'corpora', local_corpora)
    print('  done')
else:
    print('corpora already staged')

# Always take the newest database so a resumed session continues the same run.
print('copying database from Drive...')
shutil.copy2(DRIVE_ARTIFACTS / 'benchmark.sqlite', local_db)

import sqlite3
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    runs = c.execute('SELECT run_id FROM runs ORDER BY created_at DESC').fetchall()
    agg = c.execute('SELECT COUNT(*) FROM aggregate_metrics').fetchone()[0]
    gens = c.execute('SELECT COUNT(*) FROM generations').fetchone()[0]
print(f'  runs={[r[0] for r in runs]}  aggregate_metrics={agg}  generations={gens}')
if gens:
    print(f'  resuming: {gens} answers already generated will be skipped')

## 7 · Point the framework at these directories

In [ ]:
import os

# Datasets stay on Drive: they are read rarely and never written.
os.environ['RAGBENCH_DATASETS_DIR'] = str(DRIVE_DATASETS)
# Artifacts live on local disk while running, and are synced back to Drive.
os.environ['RAGBENCH_ARTIFACTS_DIR'] = str(LOCAL_ARTIFACTS)

!python -m cli datasets 2>&1 | head -12
print()
print('datasets dir :', os.environ['RAGBENCH_DATASETS_DIR'])
print('artifacts dir:', os.environ['RAGBENCH_ARTIFACTS_DIR'])

## 8 · Start the automatic Drive sync

Copies the database to Drive every 10 minutes using SQLite's online backup,
which is safe while the database is being written. Leave this running.

In [ ]:
import sqlite3
import threading
import time
from datetime import datetime

SYNC_SECONDS = 600
_stop_sync = threading.Event()


def sync_to_drive(reason='periodic'):
    # Back the live database up to Drive without interrupting writers.
    try:
        source = sqlite3.connect(f'file:{local_db}?mode=ro', uri=True)
        target = sqlite3.connect(str(DRIVE_ARTIFACTS / 'benchmark.sqlite'))
        with target:
            source.backup(target)
        source.close()
        target.close()
        stamp = datetime.now().strftime('%H:%M:%S')
        print(f'[{stamp}] synced to Drive ({reason})')
    except Exception as exc:                     # never kill the run over a sync
        print(f'sync failed ({exc}); the local database is still intact')


def _loop():
    while not _stop_sync.wait(SYNC_SECONDS):
        sync_to_drive()


threading.Thread(target=_loop, daemon=True).start()
print(f'auto-sync every {SYNC_SECONDS // 60} minutes -> {DRIVE_ARTIFACTS / "benchmark.sqlite"}')

## 9 · Generate

100 questions per dataset x 6 datasets x 3 strategies x 4 models = **7,200
answers**. Models run one at a time to avoid reloading, and every answer is
committed as it is produced, so an interrupted run resumes rather than
restarting.

In [ ]:
!python -m cli generate-all --all --models llama3.1,gemma2,mistral,qwen2.5 --limit 100 --verbose

In [ ]:
sync_to_drive('after generation')

import sqlite3
with sqlite3.connect(f'file:{local_db}?mode=ro', uri=True) as c:
    print('answers generated:', c.execute('SELECT COUNT(*) FROM generations').fetchone()[0])
    for row in c.execute(
        'SELECT model, COUNT(*), SUM(answer IS NULL) FROM generations GROUP BY model'
    ):
        print(f'  {row[0]:12s} {row[1]:6d} answers, {row[2] or 0} empty')

## 10 · Score the answers

RAGAS faithfulness with a fixed judge (`mistral`, set in `config/default.yaml`)
so every model is rated by the same rater, plus NLI hallucination detection.
Runs after generation and never during it.

In [ ]:
!python -m cli score-answers --verbose

In [ ]:
sync_to_drive('after scoring')
!python -m cli results

## 11 · Final sync and export

In [ ]:
import shutil

_stop_sync.set()
sync_to_drive('final')

# Results CSVs are small; copy the whole results directory across.
drive_results = DRIVE_ARTIFACTS / 'results'
drive_results.mkdir(parents=True, exist_ok=True)
for item in (LOCAL_ARTIFACTS / 'results').glob('*'):
    if item.is_file():
        shutil.copy2(item, drive_results / item.name)
        print('copied', item.name)

!python -m cli export
!python -m cli significance

for item in (LOCAL_ARTIFACTS / 'results').glob('*.csv'):
    shutil.copy2(item, drive_results / item.name)

print()
print('Everything is on Drive. Download artifacts/benchmark.sqlite to your')
print('laptop, drop it into C:\\AI BENCHMARK\\artifacts\\, and run:')
print('    streamlit run dashboard/app.py')

## Resuming after a disconnect

Colab caps sessions at 12 hours, and this run is longer than that. To resume:

1. Reconnect and **run every cell from the top**.
2. Step 6 pulls the partially-filled database back from Drive.
3. `generate-all` skips answers that already exist and continues.

Nothing is regenerated, so a second session costs only the work that remains.

## If the GPU runs out of memory

`gemma2` is the largest model here. On a 16 GB card, replace it with a
quantised tag and record the change as a limitation, since it breaks the
equal-precision control across models:

```python
!ollama pull gemma2:9b-instruct-q4_0
```

## Watching compute units

Units are consumed for every hour the runtime is **connected**, not every hour
the GPU is busy. Disconnect the runtime as soon as the final sync finishes.